### Implementation with 300 seconds timer

In [ ]:
import Functions.polymarket_scraper as scraper_helpers

In [ ]:
"""Infinite wall-clock loop with graceful shutdown support."""
import signal
import time
from datetime import datetime

scraper_helpers._shutdown_requested = False
signal.signal(signal.SIGTERM, scraper_helpers._request_shutdown)

if __name__ == "__main__":
    INTERVAL_SECONDS = 300

    scraper_helpers.logger.info(
        f"Initializing Wall-Clock Scraper (interval={INTERVAL_SECONDS}s)."
    )
    scraper_helpers.logger.info(
        "Send SIGTERM or press Ctrl+C to stop gracefully after current run."
    )

    try:
        while not scraper_helpers._shutdown_requested:
            now = time.time()
            time_to_wait = INTERVAL_SECONDS - (now % INTERVAL_SECONDS)
            next_run = datetime.fromtimestamp(now + time_to_wait).strftime("%H:%M:%S")
            scraper_helpers.logger.info(
                f"Sleeping {int(time_to_wait)}s until next run at {next_run}..."
            )
            time.sleep(time_to_wait)

            if not scraper_helpers._shutdown_requested:
                scraper_helpers.run_job(keyword="gold", limit_results=None)

    except KeyboardInterrupt:
        scraper_helpers.logger.info("KeyboardInterrupt received. Scraper stopped cleanly.")

    scraper_helpers.logger.info(
        f"Scraper shut down after {scraper_helpers._run_count} total runs."
    )

In [ ]:
'''test single scraper run'''
scraper_helpers.run_job(keyword="gold", limit_results=10)

In [ ]:
'''test print of market title, volume and scraping time'''


import sqlite3
import json
from pathlib import Path

DB_PATH = Path.cwd() / "Data" / "polymarket_gamma_dynamic.sqlite"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Removed 'tags' from the SELECT statement
cursor.execute("SELECT question, volume, scraped_at FROM markets LIMIT 100;")
results = cursor.fetchall()

print(f"Found {len(results)} markets. Here is the breakdown:\n")
print("-" * 50)

for row in results:
    question = row[0]
    # Format volume to look like a readable dollar amount
    volume = f"${float(row[1]):,.2f}" if row[1] else "$0.00"
    scraped_at = row[2]

    print(f"Market: {question}")
    print(f"Volume: {volume}")
    print(f"Scraped At: {scraped_at}")
    print("-" * 50)

conn.close()

In [ ]:
import sqlite3
from pathlib import Path

DB_PATH = Path.cwd() / "Data" / "polymarket_gamma_dynamic.sqlite"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
SELECT question, COUNT(DISTINCT scraped_at) AS snapshots
FROM markets
GROUP BY question
ORDER BY snapshots DESC, question
LIMIT 20;
""")

for question, snapshots in cursor.fetchall():
    print(f"{question} -> {snapshots} unique timestamps")

conn.close()